# Notebook 11 — Multiple Testing Correction

**Module:** 03 — Statistics and Probability  
**Notebook:** 11 of 18 — Track A core  
**Prerequisites:** NB10  
**Time estimate:** 90 minutes

> **Track A note:** 'You have 20,000 p-values from a differential expression analysis.
> What do you do?' — this exact question appears in Track A interviews. The answer
> to this notebook is the answer.

---
## Step 1 — Motivation

Testing 20,000 genes at α=0.05 expects 1,000 false positives by chance alone —
even if no gene is truly differentially expressed. Multiple testing correction
controls this inflation. The Benjamini-Hochberg procedure (1995) is the genomics
standard: it controls the False Discovery Rate (FDR), the fraction of significant
results that are actually false positives.

---
## Step 3 — Biological Background

**FWER vs. FDR:**
- **FWER (Family-Wise Error Rate):** probability of making *any* false positive.
  Bonferroni controls this. Very conservative — suitable for confirmatory studies
  where any false positive is unacceptable (e.g. clinical genetic testing).
- **FDR (False Discovery Rate):** *expected fraction* of discoveries that are false.
  BH controls this. Less conservative — appropriate for exploratory genomics where
  some false positives are acceptable if the overall discovery rate is controlled.

**Q-value / adjusted p-value:** the BH-corrected p-value for each test. A q-value
of 0.05 means: among all genes with q ≤ 0.05, we expect ≤5% to be false positives.

---
## Step 4 — Mathematical Explanation

**Bonferroni correction (FWER):**
$$\alpha_{\text{adjusted}} = \alpha / m$$
where $m$ = number of tests. Simple but extremely conservative.

**Benjamini-Hochberg procedure (FDR):**

1. Sort p-values: $p_{(1)} \leq p_{(2)} \leq \ldots \leq p_{(m)}$
2. Find the largest $k$ such that:
$$p_{(k)} \leq \frac{k}{m} \cdot \alpha_{\text{FDR}}$$
3. Reject $H_0$ for all hypotheses $1, \ldots, k$

**BH-adjusted p-values (q-values):**
Starting from the largest rank $m$ and working down:
$$q_{(k)} = \min\left(q_{(k+1)},\; \frac{m}{k} p_{(k)}\right)$$

**Intuition:** BH balances discovery against false positive control —
it allows some false positives (unlike Bonferroni) but keeps their fraction bounded.

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import scipy.stats as stats
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt

In [ ]:
# Cell 6.1 — Benjamini-Hochberg from scratch
def benjamini_hochberg(p_values: np.ndarray, alpha: float = 0.05) -> np.ndarray:
    """
    Benjamini-Hochberg FDR correction.

    Parameters
    ----------
    p_values : 1D array of raw p-values
    alpha    : FDR threshold

    Returns
    -------
    adjusted_p : BH-adjusted p-values (q-values)
    """
    p = np.asarray(p_values, dtype=float)
    m = len(p)
    # Sort ascending
    order = np.argsort(p)
    p_sorted = p[order]

    # Compute adjusted p-values: q(k) = min(q(k+1), m/k * p(k))
    # Work backwards from largest rank
    q_sorted = np.minimum.accumulate((m / np.arange(m, 0, -1)) * p_sorted[::-1])[::-1]
    q_sorted = np.minimum(q_sorted, 1.0)  # cap at 1

    # Return in original order
    adjusted_p = np.empty(m)
    adjusted_p[order] = q_sorted
    return adjusted_p

# Validate against statsmodels
rng = np.random.default_rng(42)
m = 1000
# 900 null, 100 true DE
null_p = rng.uniform(0, 1, 900)
de_p   = rng.beta(0.3, 10, 100)  # small p-values
all_p  = np.concatenate([null_p, de_p])

q_custom = benjamini_hochberg(all_p)
_, q_statsmodels, _, _ = multipletests(all_p, method='fdr_bh')

max_diff = np.max(np.abs(q_custom - q_statsmodels))
print(f"Max |BH custom - statsmodels|: {max_diff:.2e}  ({'OK' if max_diff < 1e-10 else 'MISMATCH'})")
print(f"Discoveries at FDR 5%: custom={( q_custom < 0.05).sum()}, statsmodels={(q_statsmodels < 0.05).sum()}")

In [ ]:
# Cell 6.2 — Applied to 500-gene expression data (from NB06)
rng2 = np.random.default_rng(0)
n_genes = 500
n_de = 50  # true DE genes
n_ctrl = n_trt = 10

true_effects = np.zeros(n_genes)
de_idx = rng2.choice(n_genes, n_de, replace=False)
true_effects[de_idx] = 1.5

ctrl_expr = rng2.normal(5.0, 1.0, (n_genes, n_ctrl))
trt_expr = ctrl_expr + true_effects[:, None] + rng2.normal(0, 0.3, (n_genes, n_trt))

p_vals = np.array([stats.ttest_ind(ctrl_expr[g], trt_expr[g])[1] for g in range(n_genes)])
log2fc = trt_expr.mean(1) - ctrl_expr.mean(1)

# Apply corrections
q_bonf = np.minimum(p_vals * n_genes, 1.0)  # Bonferroni
q_bh   = benjamini_hochberg(p_vals)

alpha = 0.05
sig_raw   = (p_vals  < alpha).sum()
sig_bonf  = (q_bonf  < alpha).sum()
sig_bh    = (q_bh    < alpha).sum()
true_pos  = np.isin(np.where(q_bh < alpha)[0], de_idx).sum()

print(f"True DE genes: {n_de}")
print(f"{'Correction':<15} {'Significant':>14} {'True positives':>17} {'Est. FDR':>12}")
print("-" * 62)
print(f"  None (raw)    {sig_raw:>12}  {'?':>15}   {'~5% per test'}")
print(f"  Bonferroni    {sig_bonf:>12}")
print(f"  BH (FDR 5%)   {sig_bh:>12}  {true_pos:>15}   {(sig_bh-true_pos)/max(sig_bh,1)*100:.1f}%")

In [ ]:
# Cell 6.3 — Q-Q plot: checking for signal in p-value distribution
# Under global H0: p-values are uniform → -log10(p) exponential
# With true signal: left tail inflated (small p-values)

observed_neg_log10_p = np.sort(-np.log10(p_vals))[::-1]
expected_neg_log10_p = -np.log10(np.linspace(1/n_genes, 1 - 1/n_genes, n_genes))
expected_neg_log10_p = np.sort(expected_neg_log10_p)[::-1]

print("Q-Q plot data prepared. See visualization cell.")
print(f"Genomic inflation factor λ = {np.median(stats.chi2.ppf(1-p_vals, df=1)) / stats.chi2.ppf(0.5, df=1):.4f}")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Volcano plot + Q-Q plot + method comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel 1: Volcano with BH threshold
ax = axes[0]
neg_log10p = -np.log10(p_vals)
sig_bh_mask = q_bh < 0.05
tp_mask = sig_bh_mask & np.isin(np.arange(n_genes), de_idx)
fp_mask = sig_bh_mask & ~np.isin(np.arange(n_genes), de_idx)

ax.scatter(log2fc[~sig_bh_mask], neg_log10p[~sig_bh_mask], s=8, color="gray", alpha=0.4)
ax.scatter(log2fc[tp_mask], neg_log10p[tp_mask], s=15, color="steelblue", alpha=0.9, label="True positive")
ax.scatter(log2fc[fp_mask], neg_log10p[fp_mask], s=15, color="tomato", alpha=0.9, label="False positive")
ax.axhline(-np.log10(q_bh[np.argsort(q_bh)][(q_bh < 0.05).sum()-1]),
           color="black", ls="--", lw=1, label="BH threshold")
ax.set_xlabel("log₂FC"); ax.set_ylabel("-log₁₀(p)")
ax.set_title("Volcano: BH-corrected discoveries")
ax.legend(fontsize=7)

# Panel 2: Q-Q plot
ax = axes[1]
ax.scatter(expected_neg_log10_p, observed_neg_log10_p, s=5, color="steelblue", alpha=0.6)
max_val = max(expected_neg_log10_p.max(), observed_neg_log10_p.max())
ax.plot([0, max_val], [0, max_val], 'k--', lw=1, label="y = x (null)")
ax.set_xlabel("Expected -log₁₀(p)"); ax.set_ylabel("Observed -log₁₀(p)")
ax.set_title("Q-Q plot: inflation above null line = real signal")
ax.legend(fontsize=8)

# Panel 3: p-value histogram
ax = axes[2]
ax.hist(p_vals, bins=40, color="steelblue", edgecolor="none", density=True)
ax.axhline(1.0, color="gray", ls="--", lw=1, label="Uniform null")
ax.set_xlabel("raw p-value"); ax.set_ylabel("Density")
ax.set_title("p-value histogram (spike near 0 = signal)")
ax.legend()

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Implement `benjamini_hochberg()` from scratch. Verify it against `statsmodels.multipletests`.
2. Apply Bonferroni and BH to the 500-gene dataset. Count true positives and estimated
   false discoveries for each. Which is more powerful? Which is more conservative?
3. What is the genomic inflation factor λ (lambda)? How is it computed? What does
   λ > 1.05 indicate about a GWAS result?
4. The BH procedure controls FDR at level α. What does 'controls FDR' mean precisely?
   Is it guaranteed to give exactly α fraction false positives?

---
## Quiz — Active Recall (Track A critical)

1. Write the Bonferroni correction formula. When is it appropriate?
2. Describe the Benjamini-Hochberg procedure in 4 steps from memory.
3. What is FDR? How is it different from FWER?
4. You have 20,000 p-values. How many false positives do you expect at α=0.05 without correction?
5. What does a q-value of 0.05 mean? Is it the same as a p-value of 0.05?

---
## Papers Referenced

- Benjamini & Hochberg (1995). DOI: 10.1111/j.2517-6161.1995.tb02031.x

---
## Reflection

**Date completed:** ____________________

> *[Could you implement BH from scratch without looking? Could you explain the difference between FDR and FWER in 30 seconds?]*

---
*Next: `12_power_analysis.ipynb`*